# Dataset v1 vs v2: balanced multi-seed comparison

Этот короткий эксперимент проверяет, устойчиво ли добавление двух train-спикеров
улучшает обобщение на прежнего validation-спикера.

Он **не декодирует видео и не запускает ResNet18 заново**. Используются уже
сохранённые frozen ResNet18 features. Для `v1` и `v2` полностью одинаковы:

- `BiGRUClassifier`;
- speaker-word balanced sampler;
- `AdamW`, learning rate и early stopping;
- validation speaker;
- набор seed: `42`, `123`, `2026`.

Test split не загружается и не используется.


## 1. Подключить Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## 2. Загрузить feature caches v1 и v2

In [ ]:
from collections import Counter
from pathlib import Path
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler


def load_torch_file(path, device='cpu'):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


drive_root = Path('/content/drive/MyDrive/lipreading_sem4')
v1_feature_root = drive_root / 'controlled_frozen_resnet18_bigru'
v2_feature_root = drive_root / 'controlled_frozen_resnet18_bigru_balanced_v2'

cache_paths = {
    'v1': {
        'train': v1_feature_root / 'dataset_v1_train_resnet18_imagenet_features.pt',
        'val': v1_feature_root / 'dataset_v1_val_resnet18_imagenet_features.pt',
    },
    'v2': {
        'train': v2_feature_root / 'dataset_v2_train_resnet18_imagenet_features.pt',
        'val': v2_feature_root / 'dataset_v2_val_resnet18_imagenet_features.pt',
    },
}

missing = [
    str(path)
    for version_paths in cache_paths.values()
    for path in version_paths.values()
    if not path.exists()
]
if missing:
    raise FileNotFoundError('Missing feature caches:\n' + '\n'.join(missing))

caches = {
    version: {
        split: load_torch_file(path)
        for split, path in version_paths.items()
    }
    for version, version_paths in cache_paths.items()
}

word_label_pairs = set(zip(
    caches['v1']['train']['words'],
    caches['v1']['train']['labels'].tolist(),
))
word_to_idx = {word: int(label) for word, label in word_label_pairs}
idx_to_word = {index: word for word, index in word_to_idx.items()}
vocab = [idx_to_word[index] for index in sorted(idx_to_word)]

device = 'cuda' if torch.cuda.is_available() else 'cpu'
output_dir = drive_root / 'balanced_multiseed_v1_v2'
output_dir.mkdir(parents=True, exist_ok=True)

print('device:', device)
print('vocab:', vocab)
for version in ('v1', 'v2'):
    print(
        version,
        'train:', caches[version]['train']['features'].shape,
        'validation:', caches[version]['val']['features'].shape,
    )
print('results:', output_dir)
print('TEST SPLIT IS NOT LOADED.')


## 3. Проверить, что validation в v1 и v2 действительно одинаковый

In [ ]:
v1_val = caches['v1']['val']
v2_val = caches['v2']['val']

def normalize_clip_path(path):
    path = str(path).replace('\\', '/')
    return path.removeprefix('02_model_inputs/')


same_paths = (
    [normalize_clip_path(path) for path in v1_val['clip_paths']]
    == [normalize_clip_path(path) for path in v2_val['clip_paths']]
)
same_words = v1_val['words'] == v2_val['words']
same_speakers = v1_val['speakers'] == v2_val['speakers']
same_cached_label_ids = torch.equal(v1_val['labels'], v2_val['labels'])
max_feature_difference = (
    v1_val['features'].float() - v2_val['features'].float()
).abs().max().item()

print('same clip order:', same_paths)
print('same words:', same_words)
print('same speakers:', same_speakers)
print('same cached numeric label ids:', same_cached_label_ids)
print('max validation feature difference:', max_feature_difference)

assert same_paths and same_words and same_speakers
assert max_feature_difference < 1e-5
print('Cached label ids differ because v1/v2 vocab orders differ.')
print('Targets will be rebuilt from words with one canonical word_to_idx.')
shared_val_cache = v2_val
print('Both versions will use the same v2 validation feature tensor.')


## 4. Dataset, balanced sampler, модель и метрики

In [ ]:
def set_reproducible_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


class FeatureSequenceDataset(Dataset):
    def __init__(self, cache, word_to_idx):
        self.features = cache['features'].float()
        self.labels = torch.tensor(
            [word_to_idx[word] for word in cache['words']],
            dtype=torch.long,
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]


class BiGRUClassifier(nn.Module):
    def __init__(self, num_classes, input_dim=512, hidden_dim=128, dropout=0.4):
        super().__init__()
        self.temporal_encoder = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, num_classes),
        )

    def forward(self, features):
        _, hidden = self.temporal_encoder(features)
        clip_features = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.classifier(clip_features)


def make_loaders(train_cache, val_cache, seed):
    group_keys = list(zip(train_cache['speakers'], train_cache['words']))
    group_counts = Counter(group_keys)
    sample_weights = torch.tensor(
        [1.0 / group_counts[key] for key in group_keys],
        dtype=torch.double,
    )
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
        generator=torch.Generator().manual_seed(seed),
    )
    train_loader = DataLoader(
        FeatureSequenceDataset(train_cache, word_to_idx),
        batch_size=32,
        sampler=sampler,
        num_workers=0,
    )
    val_loader = DataLoader(
        FeatureSequenceDataset(val_cache, word_to_idx),
        batch_size=32,
        shuffle=False,
        num_workers=0,
    )
    # Earlier v1/v2 notebooks displayed one preview batch before training.
    # Consume the same batch here so seed 42 follows the same sampler state.
    _ = next(iter(train_loader))
    return train_loader, val_loader, len(group_counts)


def classification_metrics(preds, labels, num_classes):
    matrix = np.zeros((num_classes, num_classes), dtype=int)
    for true_index, pred_index in zip(labels, preds):
        matrix[true_index, pred_index] += 1

    class_f1 = []
    class_recall = []
    for class_id in range(num_classes):
        true_positive = matrix[class_id, class_id]
        support = matrix[class_id, :].sum()
        predicted = matrix[:, class_id].sum()
        precision = true_positive / predicted if predicted else 0.0
        recall = true_positive / support if support else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        if support:
            class_f1.append(f1)
            class_recall.append(recall)

    return {
        'accuracy': float(np.trace(matrix) / matrix.sum()),
        'macro_f1': float(np.mean(class_f1)),
        'balanced_accuracy': float(np.mean(class_recall)),
        'predicted_classes': int(np.count_nonzero(matrix.sum(axis=0))),
    }


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total = 0
    preds = []
    labels = []
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            outputs = model(batch_x)
            total_loss += criterion(outputs, batch_y).item() * batch_y.size(0)
            total += batch_y.size(0)
            preds.extend(outputs.argmax(dim=1).cpu().tolist())
            labels.extend(batch_y.cpu().tolist())
    metrics = classification_metrics(preds, labels, len(vocab))
    metrics['loss'] = total_loss / total
    return metrics


## 5. Одинаковый training protocol для одного запуска

In [ ]:
def run_one_seed(version, train_cache, val_cache, seed, num_epochs=30, patience=7):
    set_reproducible_seed(seed)
    train_loader, val_loader, group_count = make_loaders(train_cache, val_cache, seed)
    set_reproducible_seed(seed)
    model = BiGRUClassifier(num_classes=len(vocab)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    best_macro_f1 = -1.0
    best_val_loss = float('inf')
    best_row = None
    best_state = None
    no_improvement = 0
    start = time.perf_counter()

    for epoch in range(1, num_epochs + 1):
        model.train()
        train_loss_sum = 0.0
        train_correct = 0
        train_total = 0

        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * batch_y.size(0)
            train_correct += (outputs.argmax(dim=1) == batch_y).sum().item()
            train_total += batch_y.size(0)

        val_metrics = evaluate(model, val_loader, criterion)
        train_loss = train_loss_sum / train_total
        train_accuracy = train_correct / train_total
        improved = (
            val_metrics['macro_f1'] > best_macro_f1
            or (
                val_metrics['macro_f1'] == best_macro_f1
                and val_metrics['loss'] < best_val_loss
            )
        )

        if improved:
            best_macro_f1 = val_metrics['macro_f1']
            best_val_loss = val_metrics['loss']
            no_improvement = 0
            best_row = {
                'dataset_version': version,
                'seed': seed,
                'best_epoch': epoch,
                'train_clips': len(train_cache['labels']),
                'speaker_word_groups': group_count,
                'train_accuracy_at_best': train_accuracy,
                'train_loss_at_best': train_loss,
                'validation_loss': val_metrics['loss'],
                'validation_accuracy': val_metrics['accuracy'],
                'validation_macro_f1': val_metrics['macro_f1'],
                'validation_balanced_accuracy': val_metrics['balanced_accuracy'],
                'predicted_classes': val_metrics['predicted_classes'],
            }
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
        else:
            no_improvement += 1

        if no_improvement >= patience:
            break

    elapsed = time.perf_counter() - start
    best_row['epochs_completed'] = epoch
    best_row['training_seconds'] = elapsed
    checkpoint_path = output_dir / f'{version}_seed{seed}_best.pt'
    torch.save({
        'model_state_dict': best_state,
        'vocab': vocab,
        'result': best_row,
        'test_used': False,
    }, checkpoint_path)

    print(
        f"{version} seed={seed} | epoch={best_row['best_epoch']} | "
        f"acc={best_row['validation_accuracy']:.4f} | "
        f"macro-F1={best_row['validation_macro_f1']:.4f} | "
        f"balanced_acc={best_row['validation_balanced_accuracy']:.4f} | "
        f"classes={best_row['predicted_classes']}/10 | {elapsed:.1f}s"
    )
    return best_row


## 6. Запустить три seed для v1 и v2

In [ ]:
SEEDS = [42, 123, 2026]
results = []

for version in ('v1', 'v2'):
    for seed in SEEDS:
        results.append(run_one_seed(
            version=version,
            train_cache=caches[version]['train'],
            val_cache=shared_val_cache,
            seed=seed,
        ))

print('completed runs:', len(results))
print('TEST SPLIT WAS NOT USED.')


## 7. Среднее, стандартное отклонение и парная разница

In [ ]:
runs_df = pd.DataFrame(results).sort_values(['dataset_version', 'seed'])
runs_path = output_dir / 'v1_v2_multiseed_runs.csv'
runs_df.to_csv(runs_path, index=False)
display(runs_df)

metric_columns = [
    'validation_accuracy',
    'validation_macro_f1',
    'validation_balanced_accuracy',
    'predicted_classes',
]
summary_df = runs_df.groupby('dataset_version')[metric_columns].agg(['mean', 'std'])
summary_df.columns = [f'{metric}_{stat}' for metric, stat in summary_df.columns]
summary_df = summary_df.reset_index()
summary_path = output_dir / 'v1_v2_multiseed_summary.csv'
summary_df.to_csv(summary_path, index=False)
display(summary_df)

paired = runs_df.pivot(index='seed', columns='dataset_version', values='validation_macro_f1')
paired['v2_minus_v1_macro_f1'] = paired['v2'] - paired['v1']
paired_path = output_dir / 'v1_v2_paired_macro_f1.csv'
paired.to_csv(paired_path)
display(paired)

figure_path = output_dir / 'v1_v2_multiseed_macro_f1.png'
fig, ax = plt.subplots(figsize=(7, 4))
for seed, row in paired.iterrows():
    ax.plot(['v1', 'v2'], [row['v1'], row['v2']], marker='o', label=f'seed {seed}')
ax.set_ylabel('validation macro-F1')
ax.set_title('Dataset v1 vs v2: paired seed comparison')
ax.legend()
fig.tight_layout()
fig.savefig(figure_path, dpi=150, bbox_inches='tight')
plt.show()

print('saved runs:', runs_path)
print('saved summary:', summary_path)
print('saved paired deltas:', paired_path)
print('saved figure:', figure_path)


## 8. Автоматическая интерпретация

In [ ]:
v1_mean = runs_df.loc[
    runs_df['dataset_version'] == 'v1', 'validation_macro_f1'
].mean()
v2_mean = runs_df.loc[
    runs_df['dataset_version'] == 'v2', 'validation_macro_f1'
].mean()
mean_delta = v2_mean - v1_mean
wins = int((paired['v2_minus_v1_macro_f1'] > 0).sum())

print('v1 mean macro-F1:', round(v1_mean, 4))
print('v2 mean macro-F1:', round(v2_mean, 4))
print('mean v2 - v1:', round(mean_delta, 4))
print('v2 wins by seed:', f'{wins}/{len(SEEDS)}')

if mean_delta > 0 and wins >= 2:
    print('DECISION: dataset v2 gives a reasonably consistent improvement.')
else:
    print('DECISION: dataset v2 has not shown a consistent improvement for this model.')
    print('Next: change the visual/temporal model or training strategy, not collect videos blindly.')

print('TEST SPLIT WAS NOT USED.')
